In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from pycontrails import Fleet

In [2]:
dff = pd.read_parquet("../data/filed/filed_trajectories_EAGWP100.parquet")
fleetf = Fleet(data=dff)
print(f"Fleet contains {fleetf.n_flights} flights")

dfo = pd.read_parquet("../data/optimised/optimised_trajectories_EAGWP100.parquet")
fleeto = Fleet(data=dfo)
print(f"Fleet contains {fleeto.n_flights} flights")

Fleet contains 4112 flights
Fleet contains 4112 flights


In [3]:
from cane.utils import mask_by_marker

mask_by_marker(fleetf, ["nox", "NOx", "O3", "CH4"])
mask_by_marker(fleeto, ["nox", "NOx", "O3", "CH4"])

dff = fleetf.dataframe
dfo = fleeto.dataframe

Marked rows filtered out for nox
Marked rows filtered out for NOx
Marked rows filtered out for O3
Marked rows filtered out for CH4
Marked rows filtered out for nox
Marked rows filtered out for NOx
Marked rows filtered out for O3
Marked rows filtered out for CH4


In [4]:
from cane.utils import mask_by_validity_range

bounds = [150, 350]
mask_by_validity_range(fleetf, ["nox", "NOx", "O3", "CH4", "H2O"], bounds)
mask_by_validity_range(fleeto, ["nox", "NOx", "O3", "CH4", "H2O"], bounds)

dff = fleetf.dataframe
dfo = fleeto.dataframe

Bounds of [150, 350] in place for nox
Bounds of [150, 350] in place for NOx
Bounds of [150, 350] in place for O3
Bounds of [150, 350] in place for CH4
Bounds of [150, 350] in place for H2O
Bounds of [150, 350] in place for nox
Bounds of [150, 350] in place for NOx
Bounds of [150, 350] in place for O3
Bounds of [150, 350] in place for CH4
Bounds of [150, 350] in place for H2O


In [5]:
dff["CO2_CoCiP"] = dff["CO2"] + dff["CoCiP"]
dfo["CO2_CoCiP"] = dfo["CO2"] + dfo["CoCiP"]

dff["Total"] = dff["CO2"] + dff["CoCiP"] + dff["NOx"] + dff["H2O"]
dfo["Total"] = dfo["CO2"] + dfo["CoCiP"] + dfo["NOx"] + dfo["H2O"]

In [6]:
from cane.utils import df_diff

# create aggregates (total)
org_sum = (
    dff[["flight_id", "fuel_burn", "nox", "co2", "CO2", "NOx", "CoCiP", "CO2_CoCiP", "Total"]]
    .groupby(["flight_id"]).sum().reset_index()
)
opt_sum = (
    dfo[["flight_id", "fuel_burn", "nox", "co2", "CO2", "NOx", "CoCiP", "CO2_CoCiP", "Total"]]
    .groupby(["flight_id"]).sum().reset_index()
)
my_diff = df_diff(org_sum, opt_sum, on=["flight_id"], keep_originals=True)
my_diff = pd.merge(
    my_diff,
    dff.groupby("flight_id").day.first(),
    on="flight_id",
)

In [7]:
def pie_and_ranking(df, cols):
    df['total'] = df[cols].sum(axis=1)

    # Compute shares (like in a pie chart)
    for col in cols:
        df[f'{col}_share'] = df[col] / df['total']

    # Compute ranking (1 = largest)
    df['ranking'] = df[cols].apply(lambda x: x.rank(ascending=False).astype(int).to_dict(), axis=1)
    df = pd.concat([df, df.ranking.apply(pd.Series).add_prefix('ranking_')], axis=1)
    return df


pie_cols = ['flight_id', 'CO2', 'CoCiP', 'NOx']
res = pie_and_ranking(
    org_sum[pie_cols].reset_index(drop=True),
    cols=['CO2', 'CoCiP', 'NOx']
)
# res = res[res["co2eq_contrail"] > 0].reset_index(drop=True)

res_opt = pie_and_ranking(
    opt_sum[pie_cols].reset_index(drop=True),
    cols=['CO2', 'CoCiP', 'NOx']
)

In [8]:
def ranking_to_items_by_position(rank_str):
    """
    Convert a ranking string to ordered item names by position.

    rank_str: string like "231"
    First digit = CO2, second = Contrail, third = NOx
    """
    items = ["CO$_2$", "Contrail", "NO$_x$"]

    # Pair each item with its rank
    item_rank_pairs = [(item, int(rank)) for item, rank in zip(items, rank_str)]

    # Sort by rank
    item_rank_pairs.sort(key=lambda x: x[1])

    # Return comma-separated names
    return ", ".join([item for item, _ in item_rank_pairs])

In [9]:
my_diff["gap"] = abs(abs(my_diff["Total_diff"]) - abs(my_diff["CO2_CoCiP_diff"]))
# my_diff[["CO2_diff", "NOx_diff", "CoCiP_diff", "Total_diff", "CO2_CoCiP_diff",
#          "gap"]].sort_values(by="gap", ascending=False).reset_index(
#     drop=True)

In [10]:
the_86 = my_diff[(my_diff["CO2_CoCiP_diff"] < 0) & (my_diff["Total_diff"] > 0)].sort_values(by="gap",
                                                                                            ascending=False).reset_index(
    drop=True)
# are now actually only 32
print(the_86.shape[0], "flights")

86 flights


In [11]:
cols = ["flight_id", "ranking_CO2", "ranking_CoCiP", "ranking_NOx"]
df = pd.merge(
    res.query("flight_id in @the_86.flight_id.unique()")[cols],
    res_opt.query("flight_id in @the_86.flight_id.unique()")[cols],
    on="flight_id",
    suffixes=("_f", "_o")
)

In [12]:
cols = ["flight_id", "ranking_CO2", "ranking_CoCiP", "ranking_NOx"]
df = pd.merge(
    res[cols],
    res_opt[cols],
    on="flight_id",
    suffixes=("_f", "_o")
)
df["filed"] = df["ranking_CO2_f"].astype(str) + df["ranking_CoCiP_f"].astype(str) + df[
    "ranking_NOx_f"].astype(str)
df["optimized"] = df["ranking_CO2_o"].astype(str) + df["ranking_CoCiP_o"].astype(str) + df[
    "ranking_NOx_o"].astype(str)

df = df.sort_values(by="filed", ascending=False).reset_index(drop=True)
# df[["origin", "destination"]]
df = df.sort_values(
    by=["ranking_CO2_f", "ranking_CoCiP_f", "ranking_NOx_f"],
    ascending=False
).reset_index(drop=True)
df["origin"] = df["filed"].apply(ranking_to_items_by_position)
df["destination"] = df["optimized"].apply(ranking_to_items_by_position)

In [13]:
cats = np.sort(np.unique(np.concatenate([df.origin.unique(), df.destination.unique()])))
# Compute total per origin
origin_totals = df.groupby("origin").size().sort_values(ascending=False).to_dict()

# Create new labels for origins including totals
origin_labels = {orig: f"{orig} ({origin_totals[orig]})" for orig in df["origin"].unique()}

df["origin_count"] = df["origin"].map(df["origin"].value_counts())
df["origin_label"] = df["origin"].map(origin_labels)

df = df.sort_values(by="origin_count", ascending=True).reset_index(drop=True)

In [14]:
# smallest/biggest savings
import re

tmp = pd.merge(
    my_diff[["flight_id", "Total_diff"]],
    df[["flight_id", "origin_count", "origin_label"]],
    on="flight_id"
).sort_values(by="origin_count", ascending=True).reset_index(drop=True)

# positive and negative
neg = tmp[tmp["Total_diff"] < 0].copy()
pos = tmp[tmp["Total_diff"] > 0].copy()

neg["bin"], bin_edges_neg = pd.qcut(neg["Total_diff"] / 1e3, q=4, retbins=True)  # 5 bins for negative values
pos["bin"], bin_edges_pos = pd.qcut(pos["Total_diff"] / 1e3, q=1, retbins=True)  # 5 bins for positive values

df_binned = pd.concat([neg, pos]).sort_index()
df_binned.bin.unique()

tmp = df_binned
bin_edges = np.concatenate([bin_edges_neg[:-1], [0], bin_edges_pos[1:]])  #.sort_index()
# tmp["value_bin"], bin_edges = qcut_split_at_zero(tmp["co2eq_total_diff"] / 1e3, q_neg=10, q_pos=2)

labels = [
    f"({round(bin_edges[i])} – {round(bin_edges[i + 1])}]"
    for i in range(len(bin_edges) - 1)
]
tmp["value_bin"] = pd.cut(
    tmp["Total_diff"] / 1e3,
    bins=bin_edges,
    labels=labels,
    include_lowest=True
)

left_labels = tmp.origin_label.unique()[::-1]
sorted_bins = sorted(tmp["value_bin"].unique(), key=lambda s: float(re.findall(r'-?\d+', s)[0]))

cats = np.concatenate([left_labels, sorted_bins])

colorDict = dict(zip([str(y) for y in cats], sns.color_palette("PuOr_r", len(cats))))
right_labels = sorted(tmp.value_bin.unique().tolist(), key=lambda s: float(re.findall(r'-?\d+', s)[0]))[::-1]

In [15]:
from cane.sankey import sankey

sankey(tmp["origin_label"], tmp["value_bin"], aspect=20,
       fontsize=12,
       colorDict=colorDict,
       rightLabels=right_labels
       )  #, leftLabels=left_labels, rightLabels=sorted_bins)  #, colorDict=colors, fontsize=12)
fig = plt.gcf()
fig.set_facecolor("w")

fig.text(
    -0.25, 0.5,  # coordinates in data space
    "Ranking of the three biggest climate forcers per flight",
    rotation=90,  # rotate 90 degrees
    va="center",  # vertical alignment (optional)
    ha="center",  # horizontal alignment (optional),
    fontsize=12
)

fig.text(
    1.1, 0.5,  # coordinates in data space
    "$\Delta$ Climate effect [tCO$_{2eq}$, EGWP100]",
    rotation=90,  # rotate 90 degrees
    va="center",  # vertical alignment (optional)
    ha="center",  # horizontal alignment (optional)
    fontsize=12
)

fig.savefig("../figures/fig09.png", dpi=300, bbox_inches="tight")  # given in [ton]

In [16]:
tmp.groupby(["value_bin", "origin_label"]).size()

# contrail, co2, nox -> 0-132: 123
# co2, nox, contrail -> 0-132: 369
# co2, contrail, nox -> 0-132: 132

print(123/4112 * 100, 369/4112*100, 132/4112 * 100)
print("3%, 9%, 3.2%")
print((123 + 369 + 132), (123 + 369 + 132) / 4112 * 100)

2.9912451361867705 8.97373540856031 3.2101167315175094
3%, 9%, 3.2%
624 15.17509727626459


In [17]:
fids = my_diff.query("CO2_CoCiP_diff < 0 and Total_diff > 0").flight_id.unique()
len(fids)

86

In [18]:
the_86.shape[0]

86

In [19]:
1617 + 1387 + 1108

4112

In [20]:
1495 + 1043 + 965

3503